# Evaluación independiente y controles de validez
Calcula métricas solo con etiquetas aprobadas. Sin el gold independiente, comprueba el protocolo y identifica las métricas no estimadas como nulas.


In [1]:
from pathlib import Path
import sys
root = Path.cwd()
if not (root / "src").exists():
    root = root.parent
assert (root / "src").exists(), "Ejecute desde la raíz o desde notebooks"
sys.path.insert(0, str(root))


In [2]:
from src.evaluate import retrieval_metrics, wilson, evaluate
assert retrieval_metrics(["incorrecto", "A"], ["A", "B"])["recall_at_5"] == .5
assert retrieval_metrics(["incorrecto", "A"], ["A", "B"])["hit_at_5"] == 1
print({"synthetic_test": "correcto", "wilson_example_8_of_10": wilson(8, 10), "note": "Ejemplo matemático sintético, no calidad del sistema"})

{'synthetic_test': 'correcto', 'wilson_example_8_of_10': [0.49015684672072335, 0.9433190520193067], 'note': 'Ejemplo matemático sintético, no calidad del sistema'}


In [3]:
from src.utils import read_jsonl, save_json, digest
from src.model import BM25
labels = root / "evaluation/gold_approved.jsonl"
corpus = root / "data/processed/chunks.jsonl"
if labels.exists():
    approved = read_jsonl(labels)
    report = evaluate(BM25(read_jsonl(corpus)), approved, digest(corpus), "test")
else:
    report = {"status": "not_estimated", "quality_metrics": None, "reason": "La entrega reporta evaluación de desarrollo y funcional; no contiene una estimación independiente con pasajes anotados por especialistas"}
save_json(root / "results/metrics/independent_evaluation_status.json", report)
report

{'status': 'not_estimated',
 'quality_metrics': None,
 'reason': 'La entrega reporta evaluación de desarrollo y funcional; no contiene una estimación independiente con pasajes anotados por especialistas'}

## Herramientas reproducibles de evaluación
La implementación incluye comparación BM25, BM25 ajustado, E5 y RRF sobre el mismo corpus, más src/answer_evaluation.py para soporte humano de afirmaciones, abstención y tiempos pareados. Consulte evaluation/README.md. El alcance reportado no contiene una estimación independiente de calidad; los casos sintéticos verifican funciones y no sustituyen anotaciones de especialistas.